In [11]:
import pandas as pd
import numpy as np
import sklearn as sk
from scipy.sparse.linalg import svds
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse.linalg import svds



In [2]:
jobs = pd.read_parquet('db/jobs.parquet')
ratings = pd.read_parquet('db/ratings.parquet')
movies = pd.read_parquet('db/movies.parquet')
actors = pd.read_parquet('db/actors.parquet')


In [3]:
movies.dtypes

primaryTitle         str
originalTitle        str
startYear        float64
genres             int64
dtype: object

In [4]:
ratings.dtypes

averageRating    float64
numVotes           int64
dtype: object

In [5]:
df = movies.merge(ratings, on='tconst', how='inner')
jobs.reset_index(inplace=True)
df = df.merge(jobs, on='tconst', how='inner')
df

In [6]:
df.set_index(['tconst', 'nconst'], inplace=True)
df

primaryTitle                originalTitle  \
tconst    nconst                                                                
tt0000009 nm0063086                   Miss Jerry                   Miss Jerry   
          nm0183823                   Miss Jerry                   Miss Jerry   
          nm1309758                   Miss Jerry                   Miss Jerry   
tt0000574 nm0846887  The Story of the Kelly Gang  The Story of the Kelly Gang   
          nm0846894  The Story of the Kelly Gang  The Story of the Kelly Gang   
...                                          ...                          ...   
tt9916730 nm4852679                       6 Gunn                       6 Gunn   
          nm9050497                       6 Gunn                       6 Gunn   
          nm7365126                       6 Gunn                       6 Gunn   
          nm1576284                       6 Gunn                       6 Gunn   
          nm4289680                       6 Gunn                       6 Gunn   

                     startYear    genres  averageRating  numVotes  ordering  
tconst    nconst                                                             
tt0000009 nm0063086     1894.0  16777216            5.3       236         1  
          nm0183823     1894.0  16777216            5.3       236         2  
          nm1309758     1894.0  16777216            5.3       236         3  
tt0000574 nm0846887     1906.0   2101250            6.0      1065         1  
          nm0846894     1906.0   2101250            6.0      1065         2  
...                        ...       ...            ...       ...       ...  
tt9916730 nm4852679     2017.0  67108864            7.0        14         4  
          nm9050497     2017.0  67108864            7.0        14         5  
          nm7365126     2017.0  67108864            7.0        14         6  
          nm1576284     2017.0  67108864            7.0        14         7  
          nm4289680     2017.0  67108864            7.0        14         8  

[2587758 rows x 7 columns]

In [7]:
df.drop(columns=['genres', 'primaryTitle', 'originalTitle', 'startYear'], inplace=True)
df['weight'] = 1 / df['ordering']
df.drop(columns=['ordering', 'numVotes'], inplace=True)
df

In [8]:
df.reset_index(inplace=True)
df

In [17]:
enc = OneHotEncoder(
    sparse_output=True,
    handle_unknown='infrequent_if_exist',
    min_frequency=50,
    dtype=np.float32
)

A = enc.fit_transform(df[['nconst']])
print(A.shape)


U, S, Vt = svds(A, k=50)

# reordenando porque svds devolve valores singulares em ordem aleatória

idx = S.argsort()[::-1]
S = S[idx]
U = U[:, idx]
Vt = Vt[idx, :]

print(S)
print(S.shape)
